In [ ]:
%cd ..

c:\Users\arik_\Documents\Dokumente\Job_Clausthal\TNTM\TNTM_Revision_TNNLS\TNTM


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
import torch

from Code.Evaluate.Metrics import score_all, get_tw_embeddings
from Code.TNTM.TNTM_bow import TNTM_bow

In [ ]:

#with open("C:\\Users\\arik_\\\Documents\\Dokumente\\Job_Clausthal\\TNTM\\TNTM_Revision_TNNLS\\TNTM\\Data\\DataOctis2\\octis_dataset_20ng.pickle", "rb") as file:
#    bow_data = pickle.load(file)

with open("C:\\Users\\arik_\\Documents\\Dokumente\\Job_Clausthal\\TNTM\\TNTM_Revision_TNNLS\\TNTM\Data\DataOctis2\\octis_dataset_20ng.pickle", "rb") as file:
    octis_dataset = pickle.load(file)

corpus = octis_dataset.get_corpus()

In [ ]:
tw_emb = get_tw_embeddings(octis_dataset)

  1%|          | 19/3350 [00:00<00:32, 102.05it/s]


KeyboardInterrupt: 

In [ ]:
with open("C:\\Users\\arik_\\Documents\\Dokumente\\Job_Clausthal\\TNTM\\TNTM_Revision_TNNLS\\TNTM\Data\DataOctis\\cleaned_embedding_df_20ng_BERT.pickle", 'rb') as f:
    embedding_df = pickle.load(f)

In [ ]:
embedding_df.sort_values(by = "word", inplace = True)

In [ ]:
embedding_ten_lis = []

for i in range(len(embedding_df)):
    embedding_ten_lis.append(embedding_df["embedding"].iloc[i])

In [ ]:
embedding_df.sort_values(by = "word", inplace = True)
embedding_ten_lis = []

embedded_words = embedding_df.index.tolist()

for i in range(len(embedding_df)):
    embedding_ten_lis.append(embedding_df["embedding"].iloc[i])
embedding_ten = torch.stack(embedding_ten_lis)

In [ ]:
vocab = embedded_words 
vocab_set = set(vocab)
corpus = [[word for word in doc if word in vocab_set] for doc in octis_dataset.get_corpus()]

In [ ]:
model = TNTM_bow(
    n_topics = 10,
    save_path = "C:\\Users\\arik_\\Documents\\Dokumente\\Job_Clausthal\\TNTM\\TNTM_Revision_TNNLS\\TNTM\\msc\\SavedResults\\model_v1.pth",
    n_dims = 11,
    n_hidden_units = 200,
    n_encoder_layers = 3,
    enc_lr = 1e-4,
    dec_lr = 1e-3,
    n_epochs = 0,
    #batch_size = 128,
    batch_size = 256,
    dropout_rate_encoder = 0.3,
    prior_variance =  0.995, 
    prior_mean = None,
    n_topwords = 200,
    device = None, 
    validation_set_size = 0.2, 
    early_stopping = True,
    n_epochs_early_stopping = 10,
    return_embeddings = False,
    eps = 1e-4,
    umap_hyperparams = {'n_neighbors': 15, 'min_dist': 0.0}
)

In [ ]:
res_all = model.fit(
    corpus = corpus, 
    vocab = vocab, 
    embeddings = embedding_ten
)

18846it [01:28, 213.39it/s]
c:\Users\arik_\Documents\Dokumente\Job_Clausthal\TNTM\TNTM_Revision_TNNLS\TNTM\Code\TNTM\TNTM_bow.py:167: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mus_init_ten = torch.tensor(mus_init).to(self.device)
c:\Users\arik_\Documents\Dokumente\Job_Clausthal\TNTM\TNTM_Revision_TNNLS\TNTM\Code\TNTM\TNTM_bow.py:168: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  L_lower_init_ten = torch.tensor(L_lower_init).to(self.device)
c:\Users\arik_\Documents\Dokumente\Job_Clausthal\TNTM\TNTM_Revision_TNNLS\TNTM\Code\TNTM\TNTM_bow.py:169: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires

In [ ]:
print(res_all)

(array([['svga', 'sx', 'dx', ..., 'government', 'germany', 'less'],
       ['sufficient', 'ideal', 'pure', ..., 'lewis', 'republic', 'zuma'],
       ['usage', 'list', 'design', ..., 'jason', 'subscribe', 'zuma'],
       ...,
       ['not', 'without', 'alone', ..., 'hair', 'hall', 'zuma'],
       ['pass', 'stretch', 'leave', ..., 'jewish', 'jeff', 'zuma'],
       ['detector', 'clock', 'filter', ..., 'lcs', 'liberal', 'zuma']],
      dtype='<U14'), array([[6.03344256e+08, 5.91693696e+08, 5.90970752e+08, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.90217504e+08, 1.89956816e+08, 1.44044544e+08, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [5.93891700e+06, 5.74932650e+06, 5.74875100e+06, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [4.92421376e+09, 4.51002470e+09, 4.20930944e+09, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [2.53619344e+08, 2.44552240e+08, 2.21507536e+08, ...,
    

In [ ]:
octis_dataset._Dataset__corpus = corpus
octis_dataset._Dataset__vocab = vocab

In [ ]:
len(corpus)


18846

In [ ]:
evaluation_result = score_all(
    dataset = octis_dataset,
    tw_emb=tw_emb,
    n_words=10,
    result = {'topics': res_all[0], 
              "topic-word-matrix": res_all[1]},
    n_topics = 10,
    reference_corpus = "20NG"
)

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\arik_\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arik_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\arik_\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arik_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
  0%|          | 0/5 [00:00<?, ?it/s]